# Lorenz-1960 PINN (FYDP-2)

Physics-Informed Neural Network that solves the Lorenz-1960 ODE system.
Hard initial condition (trial solution), residual-only loss, Adam with
polynomial LR decay then optional L-BFGS, Latin-hypercube collocation.
Validated against the locked RK4/SciPy baseline.

Runs on **Kaggle / Colab** (GPU used automatically if available).

## Setup
On **Colab**, clone the repo first (uncomment the two lines below). On
**Kaggle**, add the repo as a dataset/utility script and start from its root.
Locally, just run it in place — the cell below finds the repository root,
switches to it, and puts `src/` on the import path.

In [ ]:
# Colab only (uncomment):
# !git clone https://github.com/ihmorol/lorenz1960-pinn.git
# %cd lorenz1960-pinn

import os, sys, pathlib
root = pathlib.Path.cwd()
if not (root / 'src' / 'fydp2').exists() and (root.parent / 'src' / 'fydp2').exists():
    root = root.parent          # notebook opened from notebooks/
os.chdir(root)
sys.path.insert(0, str(root / 'src'))
print('repo root:', root)

: 

In [ ]:
from fydp2.config import Config, reference_trajectory
from fydp2.train import train, save_results, predict, get_device
print('device:', get_device())

## Configure
Baseline architecture is 4x60 tanh. Change any field here without editing
code. For a quick CPU test reduce `epochs` (e.g. `Config(epochs=2000)`);
on a GPU the default is fine.

In [ ]:
cfg = Config()          # e.g. Config(epochs=2000) for a fast CPU run
cfg

## Train

In [ ]:
model, history = train(cfg)
print('final residual loss: %.3e' % history.loss[-1])

## Evaluate vs the locked baseline

In [ ]:
import numpy as np
metrics = save_results(model, history, cfg)   # writes results/fydp2/
t, ys = reference_trajectory(cfg)
pred = predict(model, t)
rel = np.abs(pred[-1] - ys[-1]) / np.abs(ys[-1])
print('final-state relative error:', dict(zip('xyz', rel.round(6))))
metrics

## Figures

`save_results` already wrote the full suite to `results/fydp2/`. To rebuild a
single panel interactively, call the matching `fydp2.figures.fig_*` function on
`figures.RunArtifacts` from `train.collect_artifacts(model, history, cfg)`.

In [ ]:
from IPython.display import Image, display

figs = cfg.results_path / 'figures'
display(Image(str(cfg.results_path / 'results.png')))
for name in ('training_dynamics', 'gradient_diagnostics', 'collocation_points',
             'solution_vs_reference', 'error_analysis', 'phase_portraits',
             'metrics_summary', 'physics_residual', 'invariant_drift'):
    display(Image(str(figs / f'{name}.png')))